In [ ]:
import os
import glob
import pandas as pd
from datetime import timedelta

# Define which columns to melt
price_band_cols = [f"PRICEBAND{i}" for i in range(1, 11)]

def melt_one_price_file(input_path, output_dir):
    """
    Reads one Parquet file (price bids) into memory,
    melts the PRICEBAND columns into a single column,
    and saves the melted data to a new Parquet file in output_dir.
    """
    try:
        # 1) Read the single Parquet file into a pandas DataFrame
        df = pd.read_parquet(input_path)
        
        # 2) Identify all other columns as id_vars
        id_vars_cols = [col for col in df.columns if col not in price_band_cols]
        
        # 3) Melt the DataFrame
        df_long = pd.melt(
            df,
            id_vars=id_vars_cols,
            value_vars=price_band_cols,
            var_name="BIDBAND",
            value_name="BIDPRICE"
        )
        
        # 4) Convert "BIDBAND" from e.g. "PRICEBAND1" -> integer 1
        df_long["BIDBAND"] = df_long["BIDBAND"].str.extract(r"(\d+)").astype(int)
        
        # 5) (Optional) Convert other columns to consistent dtypes
        #    Adjust column names if they differ in your data
        if "BIDTYPE" in df_long.columns:
            df_long["BIDTYPE"] = df_long["BIDTYPE"].astype(str)
            
        if "DUID" in df_long.columns:
            df_long["DUID"] = df_long["DUID"].astype(str)
            
        if "SETTLEMENTDATE" in df_long.columns:
            df_long["SETTLEMENTDATE"] = pd.to_datetime(df_long["SETTLEMENTDATE"])
            # Add 4 hours, 5 minutes offset for APPLICABLEFROM
            df_long["APPLICABLEFROM"] = df_long["SETTLEMENTDATE"] + timedelta(hours=4, minutes=5)
            df_long["APPLICABLEFROM"] = pd.to_datetime(df_long["APPLICABLEFROM"])
        
        # 6) Construct the output file path
        base_name = os.path.basename(input_path)
        melted_name = base_name.replace(".parquet", "_melted.parquet")
        output_path = os.path.join(output_dir, melted_name)
        
        # 7) Save melted data to a new Parquet file
        df_long.to_parquet(output_path, index=False)
        print(f"Melted file saved to {output_path}")
        return True
    except Exception as e:
        print(f"Error processing {input_path}: {str(e)}")
        return False

def main():
    input_dir = "/Volumes/T7/bid-price-data-excluding-energy-B2"  # Where your price Parquet files live
    output_dir = "/Volumes/T7/bid-price-melted-files"  # Where to save melted files
    os.makedirs(output_dir, exist_ok=True)
    
    # Gather Parquet files
    parquet_files = sorted(glob.glob(os.path.join(input_dir, "*.parquet")))
    
    # Exclude hidden macOS files
    parquet_files = [f for f in parquet_files if not os.path.basename(f).startswith("._")]
    print(f"Found {len(parquet_files)} Parquet files in {input_dir}.")
    
    # Process each file individually
    successful = 0
    for i, file_path in enumerate(parquet_files):
        print(f"Processing file {i+1}/{len(parquet_files)}: {os.path.basename(file_path)}")
        if melt_one_price_file(file_path, output_dir):
            successful += 1
        
        # Print progress every 10 files
        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{len(parquet_files)} files, {successful} successful")
    
    print(f"All price files processed. Successfully melted {successful} out of {len(parquet_files)} files.")
    
    # Verify output files
    output_files = glob.glob(os.path.join(output_dir, "*_melted.parquet"))
    print(f"\nVerified {len(output_files)} melted files in the output directory.")
    
    if len(output_files) > 0:
        # Optionally, load and print a sample from the first output file
        try:
            sample_file = output_files[0]
            sample_df = pd.read_parquet(sample_file)
            print(f"\nSample from first melted file ({os.path.basename(sample_file)}):")
            print(sample_df.head())
            print(f"Sample file shape: {sample_df.shape}")
        except Exception as e:
            print(f"Error reading sample file: {str(e)}")

if __name__ == "__main__":
    main()